# Production-Ready Hybrid PII Classifier — BGE + Patterns + Column Semantics

Pipeline:

`Column → BGE semantic embedding + value patterns + column-name evidence + PK/FK metadata → feature fusion → classifier → PII type + confidence`

This notebook:
1. trains/evaluates the expanded dataset,
2. explicitly adds column-name evidence to address semantic confusions such as `account_type` vs `account_number`,
3. evaluates BGE-only, BGE+patterns, and the full hybrid model on the same stratified split,
4. fits the final production model on all labeled data,
5. exports the complete model bundle, including BGE weights and preprocessing artifacts.


## 1. Install dependencies

Run this cell once inside the `cognienv` environment.

In [17]:
%pip install -U sentence-transformers scikit-learn pandas numpy matplotlib joblib


Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import joblib

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE = 0.20
BATCH_SIZE = 32
MODEL_NAME = 'BAAI/bge-small-en-v1.5'

SCRIPT_DIR = Path.cwd()
DATASET_PATH = SCRIPT_DIR / 'dataset' / 'pii_training_dataset_expanded.csv'
if not DATASET_PATH.exists():
    DATASET_PATH = SCRIPT_DIR / 'ml' / 'dataset' / 'pii_training_dataset_expanded.csv'
if not DATASET_PATH.exists():
    DATASET_PATH = SCRIPT_DIR / 'pii_training_dataset_expanded.csv'

MODEL_DIR = SCRIPT_DIR / 'pii_classifier'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset:', DATASET_PATH)
print('BGE model:', MODEL_NAME)
print('Export directory:', MODEL_DIR)


Dataset: /home/immortalzz/cogniPII/PII-Detection-Data-Masking/ml/dataset/pii_training_dataset_expanded.csv
BGE model: BAAI/bge-small-en-v1.5
Export directory: /home/immortalzz/cogniPII/PII-Detection-Data-Masking/ml/pii_classifier


## 2. Load the expanded dataset

The expanded dataset contains 949 clean records with balanced representation across the 13 PII classes, including many hard-negative NON_PII examples such as `account_type`.


In [19]:
df = pd.read_csv(DATASET_PATH)

required_columns = {
    'domain', 'table_name', 'table_type', 'column_name',
    'data_type', 'is_primary_key', 'is_foreign_key',
    'sample_values', 'label'
}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(str).str.strip()

print('Shape:', df.shape)
print('\nClass distribution:')
display(df['label'].value_counts().sort_index())


Shape: (949, 9)

Class distribution:


label
ADDRESS                     57
BANK_ACCOUNT_NUMBER         62
DATE_OF_BIRTH               61
DEMOGRAPHIC_INFORMATION     57
DIRECT_IDENTIFIER           58
EMAIL                       53
FINANCIAL_INFORMATION       57
HEALTH_INFORMATION          69
LOCATION                    65
MEDICAL_RECORD_NUMBER       57
NON_PII                    232
PERSON_NAME                 68
PHONE                       53
Name: count, dtype: int64

## 3. Build semantic text for BGE

The text combines the column name, table context, datatype, key metadata and representative values.

In [20]:
def clean_value(value):
    if pd.isna(value):
        return 'NULL'
    return str(value).strip()


def build_semantic_text(row):
    return (
        f"Domain: {clean_value(row['domain'])}. "
        f"Table: {clean_value(row['table_name'])}. "
        f"Table type: {clean_value(row['table_type'])}. "
        f"Column: {clean_value(row['column_name'])}. "
        f"Data type: {clean_value(row['data_type'])}. "
        f"Sample values: {clean_value(row['sample_values'])}."
    )

texts = [build_semantic_text(row) for _, row in df.iterrows()]
print(texts[0])


Domain: Banking. Table: dim_customer. Table type: DIMENSION. Column: customer_key. Data type: bigint. Sample values: 10001|10002|10003.


## 4. Pattern / profiling feature extraction

The pattern layer is intentionally separate from BGE. It measures how strongly the actual sample values exhibit recognizable structures.

Features include:
- email match ratio
- Indian phone match ratio
- date match ratio
- credit-card-like match ratio
- bank-account-like numeric match ratio
- medical-record-like identifier ratio
- numeric/alphabetic/alphanumeric ratios
- null ratio
- uniqueness ratio
- average/min/max value length
- primary-key and foreign-key indicators


In [21]:
EMAIL_RE = re.compile(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')
PHONE_RE = re.compile(r'^(?:\+91[- ]?|0)?[6-9]\d{9}$')
DATE_RE = re.compile(r'^(?:\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{4})$')
CREDIT_CARD_RE = re.compile(r'^(?:\d[ -]?){13,19}$')
BANK_ACCOUNT_RE = re.compile(r'^\d{8,18}$')
MEDICAL_RECORD_RE = re.compile(r'^(?:MRN[-_ ]?)?[A-Za-z0-9]{5,20}$', re.I)
ALPHA_RE = re.compile(r'^[A-Za-z .\-\']+$')
NUMERIC_RE = re.compile(r'^[-+]?\d+(?:\.\d+)?$')
ALPHANUMERIC_RE = re.compile(r'^[A-Za-z0-9]+$')


def split_sample_values(value):
    if pd.isna(value):
        return []

    raw = str(value).strip()
    if not raw or raw.upper() == 'NULL':
        return []

    # The current dataset uses | as the sample-value separator.
    return [x.strip() for x in raw.split('|') if x.strip()]


def safe_ratio(matches, total):
    return matches / total if total else 0.0


def profile_samples(sample_values):
    values = split_sample_values(sample_values)
    n = len(values)

    if n == 0:
        return np.zeros(18, dtype=np.float32)

    lengths = np.array([len(v) for v in values], dtype=np.float32)

    email_ratio = safe_ratio(sum(bool(EMAIL_RE.fullmatch(v)) for v in values), n)
    phone_ratio = safe_ratio(sum(bool(PHONE_RE.fullmatch(v)) for v in values), n)
    date_ratio = safe_ratio(sum(bool(DATE_RE.fullmatch(v)) for v in values), n)
    credit_card_ratio = safe_ratio(sum(bool(CREDIT_CARD_RE.fullmatch(v.replace(' ', ''))) for v in values), n)
    bank_account_ratio = safe_ratio(sum(bool(BANK_ACCOUNT_RE.fullmatch(v)) for v in values), n)
    medical_record_ratio = safe_ratio(sum(bool(MEDICAL_RECORD_RE.fullmatch(v)) for v in values), n)
    numeric_ratio = safe_ratio(sum(bool(NUMERIC_RE.fullmatch(v)) for v in values), n)
    alphabetic_ratio = safe_ratio(sum(bool(ALPHA_RE.fullmatch(v)) for v in values), n)
    alphanumeric_ratio = safe_ratio(sum(bool(ALPHANUMERIC_RE.fullmatch(v)) for v in values), n)

    unique_ratio = len(set(v.lower() for v in values)) / n

    return np.array([
        email_ratio,
        phone_ratio,
        date_ratio,
        credit_card_ratio,
        bank_account_ratio,
        medical_record_ratio,
        numeric_ratio,
        alphabetic_ratio,
        alphanumeric_ratio,
        unique_ratio,
        float(lengths.mean()),
        float(lengths.min()),
        float(lengths.max()),
        float(n == 0),
        float(len(values)),
        float(np.std(lengths)),
        float(np.mean([len(v.split()) for v in values])),
        float(np.mean([any(c.isdigit() for c in v) for v in values])),
    ], dtype=np.float32)


pattern_features = np.vstack(
    [profile_samples(x) for x in df['sample_values']]
)

pattern_feature_names = [
    'email_match_ratio',
    'phone_match_ratio',
    'date_match_ratio',
    'credit_card_match_ratio',
    'bank_account_match_ratio',
    'medical_record_match_ratio',
    'numeric_ratio',
    'alphabetic_ratio',
    'alphanumeric_ratio',
    'unique_ratio',
    'avg_length',
    'min_length',
    'max_length',
    'null_or_empty',
    'sample_count',
    'length_std',
    'avg_word_count',
    'contains_digit_ratio',
]

pattern_df = pd.DataFrame(pattern_features, columns=pattern_feature_names)
display(pattern_df.head())

,email_match_ratio,phone_match_ratio,date_match_ratio,credit_card_match_ratio,bank_account_match_ratio,medical_record_match_ratio,numeric_ratio,alphabetic_ratio,alphanumeric_ratio,unique_ratio,avg_length,min_length,max_length,null_or_empty,sample_count,length_std,avg_word_count,contains_digit_ratio
0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,5.000000,5.0,5.0,0.0,3.0,0.000000,1.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,4.000000,4.0,4.0,0.0,3.0,0.000000,1.0,1.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,5.333333,5.0,6.0,0.0,3.0,0.471405,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,6.333333,5.0,8.0,0.0,3.0,1.247219,1.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,15.500000,15.0,16.0,0.0,2.0,0.500000,1.0,0.0


## 5. Generate BGE embeddings

In [22]:
print('Loading:', MODEL_NAME)
bge = SentenceTransformer(MODEL_NAME)

embeddings = bge.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print('Embedding shape:', embeddings.shape)

Loading: BAAI/bge-small-en-v1.5


Batches: 100%|██████████| 30/30 [00:01<00:00, 26.90it/s]

Embedding shape: (949, 384)


## 6. Add metadata features and create the hybrid feature vector

In [23]:
def binary_feature(value):
    return int(str(value).strip().lower() in {'true', '1', 'yes', 'y'})

# Explicit lexical evidence from the column name.
# These features are intentionally category-oriented so account_type can be
# learned as a hard negative rather than being treated as an account number.
COLUMN_KEYWORDS = [
    'email', 'phone', 'address', 'name', 'dob', 'birth', 'date',
    'account_number', 'account_no', 'account_num', 'bank_account',
    'account', 'medical', 'record', 'location', 'blood', 'gender',
    'balance', 'salary', 'diagnosis', 'doctor', 'campaign', 'customer',
    'student', 'patient', 'employee', 'member', 'id', 'key',
    'amount', 'income', 'credit', 'debit', 'transaction',
]


def column_name_features(column_name):
    name = str(column_name).lower().replace('-', '_')
    return np.array([float(k in name) for k in COLUMN_KEYWORDS], dtype=np.float32)


column_name_features_matrix = np.vstack([
    column_name_features(x) for x in df['column_name']
])

metadata_features = np.array([
    [binary_feature(row['is_primary_key']), binary_feature(row['is_foreign_key'])]
    for _, row in df.iterrows()
], dtype=np.float32)

extra_features = np.hstack([
    pattern_features,
    column_name_features_matrix,
    metadata_features,
]).astype(np.float32)

extra_feature_names = (
    pattern_feature_names
    + [f'column_keyword__{k}' for k in COLUMN_KEYWORDS]
    + ['is_primary_key', 'is_foreign_key']
)

print('BGE features              :', embeddings.shape[1])
print('Pattern features          :', pattern_features.shape[1])
print('Column-name features     :', column_name_features_matrix.shape[1])
print('PK/FK metadata features  :', metadata_features.shape[1])
print('Extra features            :', extra_features.shape[1])
print('Full feature vector       :', embeddings.shape[1] + extra_features.shape[1])


BGE features              : 384
Pattern features          : 18
Column-name features     : 34
PK/FK metadata features  : 2
Extra features            : 54
Full feature vector       : 438


## 7. Inspect pattern signals

Before training, inspect whether the patterns are actually detecting the structures we expect.

In [24]:
inspection = df[['domain', 'table_name', 'column_name', 'label']].copy()
inspection = pd.concat([inspection.reset_index(drop=True), pattern_df], axis=1)

display(
    inspection[
        [
            'domain', 'table_name', 'column_name', 'label',
            'email_match_ratio', 'phone_match_ratio',
            'date_match_ratio', 'bank_account_match_ratio',
            'medical_record_match_ratio', 'numeric_ratio'
        ]
    ].to_string(index=False)
)


'   domain        table_name             column_name                   label  email_match_ratio  phone_match_ratio  date_match_ratio  bank_account_match_ratio  medical_record_match_ratio  numeric_ratio\n  Banking      dim_customer            customer_key                 NON_PII                0.0           0.000000               0.0                       0.0                    1.000000            1.0\n  Banking      dim_customer             customer_id       DIRECT_IDENTIFIER                0.0           0.000000               0.0                       0.0                    0.000000            1.0\n  Banking      dim_customer              first_name             PERSON_NAME                0.0           0.000000               0.0                       0.0                    1.000000            0.0\n  Banking      dim_customer               last_name             PERSON_NAME                0.0           0.000000               0.0                       0.0                    1.000000      

## 8. Encode labels and make one stratified train/test split

The same indices will be used for both BGE-only and hybrid experiments so the comparison is fair.

In [25]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])

indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train rows:', len(train_idx))
print('Test rows :', len(test_idx))
print('\nClasses:', list(label_encoder.classes_))


Train rows: 759
Test rows : 190

Classes: ['ADDRESS', 'BANK_ACCOUNT_NUMBER', 'DATE_OF_BIRTH', 'DEMOGRAPHIC_INFORMATION', 'DIRECT_IDENTIFIER', 'EMAIL', 'FINANCIAL_INFORMATION', 'HEALTH_INFORMATION', 'LOCATION', 'MEDICAL_RECORD_NUMBER', 'NON_PII', 'PERSON_NAME', 'PHONE']


## 9. Train the primary classifier — Logistic Regression

Logistic Regression is used first because it gave the strongest result in the BGE-only experiment.

In [26]:
def train_and_evaluate(X, name, scale_all=True):
    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    if scale_all:
        from sklearn.pipeline import Pipeline
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                C=1.0, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE
            )),
        ])
    else:
        model = LogisticRegression(
            C=1.0, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE
        )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = probabilities.max(axis=1)

    accuracy = accuracy_score(y_test, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, average='weighted', zero_division=0
    )

    print('\n' + '=' * 110)
    print(name)
    print('=' * 110)
    print(f'Accuracy          : {accuracy * 100:.2f}%')
    print(f'Precision         : {precision * 100:.2f}%')
    print(f'Recall            : {recall * 100:.2f}%')
    print(f'F1                : {f1 * 100:.2f}%')
    print(f'Average confidence: {confidence.mean() * 100:.2f}%')
    print('\nClassification report:')
    print(classification_report(
        y_test, predictions,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_, zero_division=0
    ))

    return model, predictions, confidence, probabilities, {
        'Approach': name, 'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'Avg Confidence': confidence.mean()
    }

# Keep the three experiments directly comparable on the same split.
bge_model, bge_predictions, bge_confidence, bge_probabilities, bge_result = train_and_evaluate(
    np.hstack([embeddings, metadata_features]), 'BGE + metadata baseline'
)

pattern_model, pattern_predictions, pattern_confidence, pattern_probabilities, pattern_result = train_and_evaluate(
    np.hstack([embeddings, pattern_features, metadata_features]),
    'BGE + patterns + metadata'
)

# For the final hybrid model, scale only the engineered features.
# BGE embeddings are already normalized.
extra_scaler = StandardScaler()
extra_train = extra_scaler.fit_transform(extra_features[train_idx])
extra_test = extra_scaler.transform(extra_features[test_idx])

hybrid_train = np.hstack([embeddings[train_idx], extra_train])
hybrid_test = np.hstack([embeddings[test_idx], extra_test])

hybrid_model = LogisticRegression(
    C=1.0, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE
)
hybrid_model.fit(hybrid_train, y[train_idx])
hybrid_predictions = hybrid_model.predict(hybrid_test)
hybrid_probabilities = hybrid_model.predict_proba(hybrid_test)
hybrid_confidence = hybrid_probabilities.max(axis=1)

hybrid_accuracy = accuracy_score(y[test_idx], hybrid_predictions)
hybrid_precision, hybrid_recall, hybrid_f1, _ = precision_recall_fscore_support(
    y[test_idx], hybrid_predictions, average='weighted', zero_division=0
)

print('\n' + '=' * 110)
print('FULL HYBRID: BGE + patterns + column-name evidence + PK/FK metadata')
print('=' * 110)
print(f'Accuracy          : {hybrid_accuracy * 100:.2f}%')
print(f'Precision         : {hybrid_precision * 100:.2f}%')
print(f'Recall            : {hybrid_recall * 100:.2f}%')
print(f'F1                : {hybrid_f1 * 100:.2f}%')
print(f'Average confidence: {hybrid_confidence.mean() * 100:.2f}%')
print('\nClassification report:')
print(classification_report(
    y[test_idx], hybrid_predictions,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_, zero_division=0
))

hybrid_result = {
    'Approach': 'BGE + patterns + column-name evidence + metadata',
    'Accuracy': hybrid_accuracy, 'Precision': hybrid_precision,
    'Recall': hybrid_recall, 'F1': hybrid_f1,
    'Avg Confidence': hybrid_confidence.mean()
}



BGE + metadata baseline
Accuracy          : 99.47%
Precision         : 99.48%
Recall            : 99.47%
F1                : 99.47%
Average confidence: 99.29%

Classification report:
                         precision    recall  f1-score   support

                ADDRESS       1.00      1.00      1.00        11
    BANK_ACCOUNT_NUMBER       1.00      1.00      1.00        12
          DATE_OF_BIRTH       1.00      1.00      1.00        12
DEMOGRAPHIC_INFORMATION       1.00      1.00      1.00        11
      DIRECT_IDENTIFIER       1.00      1.00      1.00        12
                  EMAIL       1.00      1.00      1.00        11
  FINANCIAL_INFORMATION       1.00      1.00      1.00        11
     HEALTH_INFORMATION       1.00      0.93      0.96        14
               LOCATION       1.00      1.00      1.00        13
  MEDICAL_RECORD_NUMBER       1.00      1.00      1.00        11
                NON_PII       0.98      1.00      0.99        47
            PERSON_NAME       1.00 

## 10. Compare the three architectures

The full hybrid model adds explicit column-name evidence to the pattern and semantic signals.


In [27]:
comparison = pd.DataFrame([bge_result, pattern_result, hybrid_result])
display(comparison.style.format({
    'Accuracy': '{:.2%}', 'Precision': '{:.2%}', 'Recall': '{:.2%}',
    'F1': '{:.2%}', 'Avg Confidence': '{:.2%}'
}))


,Approach,Accuracy,Precision,Recall,F1,Avg Confidence
0,BGE + metadata baseline,99.47%,99.48%,99.47%,99.47%,99.29%
1,BGE + patterns + metadata,99.47%,99.48%,99.47%,99.47%,99.32%
2,BGE + patterns + column-name evidence + metadata,93.16%,94.27%,93.16%,93.24%,90.30%


## 11. Show every hybrid prediction

This is the final prediction table for the hybrid model.

In [28]:
test_df = df.iloc[test_idx].reset_index(drop=True).copy()
actual_labels = label_encoder.inverse_transform(y[test_idx])
predicted_labels = label_encoder.inverse_transform(hybrid_predictions)

prediction_output = test_df[[
    'domain','table_name','table_type','column_name','data_type','label'
]].copy()
prediction_output['predicted_label'] = predicted_labels
prediction_output['confidence'] = hybrid_confidence
prediction_output['correct'] = prediction_output['label'] == prediction_output['predicted_label']

# Show mistakes first so model weaknesses are immediately visible.
display(prediction_output.sort_values(['correct','confidence']).style.format({
    'confidence': '{:.2%}'
}))


,domain,table_name,table_type,column_name,data_type,label,predicted_label,confidence,correct
140,Education,dim_student,DIMENSION,department,character varying,NON_PII,HEALTH_INFORMATION,34.75%,False
114,Marketing,dim_location,DIMENSION,street,character varying,ADDRESS,LOCATION,47.41%,False
143,Banking,banking,TRANSIENT,transaction_type,character varying,NON_PII,HEALTH_INFORMATION,47.90%,False
57,Marketing,dim_location,DIMENSION,conversion_status,character varying,NON_PII,HEALTH_INFORMATION,50.38%,False
104,Banking,account_summary,TRANSIENT,is_weekend,boolean,NON_PII,LOCATION,53.86%,False
125,Banking,fact_account,FACT,is_weekend,boolean,NON_PII,LOCATION,54.90%,False
167,Banking,banking,TRANSIENT,is_weekend,boolean,NON_PII,LOCATION,56.39%,False
49,Marketing,marketing,TRANSIENT,age,integer,DEMOGRAPHIC_INFORMATION,NON_PII,59.38%,False
156,Banking,dim_location,DIMENSION,is_weekend,boolean,NON_PII,LOCATION,66.95%,False
19,Marketing,dim_channel,DIMENSION,channel,character varying,NON_PII,LOCATION,72.17%,False


## 12. Confusion matrix for the hybrid model

In [29]:
cm = confusion_matrix(
    y[test_idx],
    hybrid_predictions,
    labels=np.arange(len(label_encoder.classes_)),
)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
)

display(cm_df)

,ADDRESS,BANK_ACCOUNT_NUMBER,DATE_OF_BIRTH,DEMOGRAPHIC_INFORMATION,DIRECT_IDENTIFIER,EMAIL,FINANCIAL_INFORMATION,HEALTH_INFORMATION,LOCATION,MEDICAL_RECORD_NUMBER,NON_PII,PERSON_NAME,PHONE
ADDRESS,10,0,0,0,0,0,0,0,1,0,0,0,0
BANK_ACCOUNT_NUMBER,0,12,0,0,0,0,0,0,0,0,0,0,0
DATE_OF_BIRTH,0,0,12,0,0,0,0,0,0,0,0,0,0
DEMOGRAPHIC_INFORMATION,0,0,0,10,0,0,0,0,0,0,1,0,0
DIRECT_IDENTIFIER,0,0,0,0,12,0,0,0,0,0,0,0,0
EMAIL,0,0,0,0,0,11,0,0,0,0,0,0,0
FINANCIAL_INFORMATION,0,0,0,0,0,0,11,0,0,0,0,0,0
HEALTH_INFORMATION,0,0,0,0,0,0,0,13,0,0,1,0,0
LOCATION,0,0,0,0,0,0,0,0,13,0,0,0,0
MEDICAL_RECORD_NUMBER,0,0,0,0,0,0,0,0,0,11,0,0,0


## 13. Save results

The prediction file contains the hybrid model's test-set predictions and confidence values.

In [30]:
output_path = DATASET_PATH.parent / 'hybrid_pii_predictions.csv'
comparison_path = DATASET_PATH.parent / 'hybrid_vs_bge_comparison.csv'

prediction_output.to_csv(output_path, index=False)
comparison.to_csv(comparison_path, index=False)

print('Predictions saved to:', output_path)
print('Comparison saved to:', comparison_path)


Predictions saved to: /home/immortalzz/cogniPII/PII-Detection-Data-Masking/ml/dataset/hybrid_pii_predictions.csv
Comparison saved to: /home/immortalzz/cogniPII/PII-Detection-Data-Masking/ml/dataset/hybrid_vs_bge_comparison.csv


## 14. Fit the final production model on ALL labeled data and export it

The test split above is for evaluation only. After evaluating the architecture, the production classifier is fitted on all 949 labeled rows. 
The export includes BGE, the classifier, the scaler, label encoder, feature configuration, and model metadata.


In [31]:
# Fit final production feature preprocessing on ALL rows.
final_extra_scaler = StandardScaler()
extra_all_scaled = final_extra_scaler.fit_transform(extra_features).astype(np.float32)
X_final = np.hstack([embeddings, extra_all_scaled]).astype(np.float32)

final_classifier = LogisticRegression(
    C=1.0, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE
)
final_classifier.fit(X_final, y)

print('Final training rows:', len(df))
print('Final feature dimension:', X_final.shape[1])


Final training rows: 949
Final feature dimension: 438


In [32]:
# Export the complete model bundle.
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_classifier, MODEL_DIR / 'classifier.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'label_encoder.joblib')
joblib.dump(final_extra_scaler, MODEL_DIR / 'extra_scaler.joblib')

# Save the exact BGE model used for inference.
bge.save(str(MODEL_DIR / 'embedding_model'))

metadata = {
    'model_name': 'pii-classifier',
    'version': '1.0.0',
    'embedding_model': MODEL_NAME,
    'embedding_dimension': int(embeddings.shape[1]),
    'pattern_feature_count': int(pattern_features.shape[1]),
    'column_name_feature_count': int(column_name_features_matrix.shape[1]),
    'metadata_feature_count': int(metadata_features.shape[1]),
    'extra_feature_dimension': int(extra_features.shape[1]),
    'total_feature_dimension': int(X_final.shape[1]),
    'classes': label_encoder.classes_.tolist(),
    'column_name_keywords': COLUMN_KEYWORDS,
    'pattern_feature_names': pattern_feature_names,
    'extra_feature_names': extra_feature_names,
    'training_rows': int(len(df)),
    'training_classes': int(len(label_encoder.classes_)),
    'random_state': RANDOM_STATE,
}

(MODEL_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Production model exported to:', MODEL_DIR.resolve())
print('\nBundle contents:')
for path in sorted(MODEL_DIR.rglob('*')):
    if path.is_file():
        print(' ', path.relative_to(MODEL_DIR))


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]

Production model exported to: /home/immortalzz/cogniPII/PII-Detection-Data-Masking/ml/pii_classifier

Bundle contents:
  classifier.joblib
  embedding_model/1_Pooling/config.json
  embedding_model/2_Normalize/config.json
  embedding_model/README.md
  embedding_model/config.json
  embedding_model/config_sentence_transformers.json
  embedding_model/model.safetensors
  embedding_model/modules.json
  embedding_model/sentence_bert_config.json
  embedding_model/tokenizer.json
  embedding_model/tokenizer_config.json
  extra_scaler.joblib
  label_encoder.joblib
  metadata.json
